#### DATA PREPARATION

##### 02.1 DOCUMENTAÇÃO DO DATA PREPARATION

###### Objetivo

O **Data Preparation** é responsável por validar, identificar e organizar as características técnicas do DataFrame recebido do Data Ingestion, disponibilizando uma estrutura padronizada para os Profiles de qualidade de dados.

Essa etapa funciona como uma camada intermediária entre a ingestão e as análises, permitindo que os Profiles utilizem informações estruturais compartilhadas sem repetir procedimentos de identificação e classificação.

O componente não aplica regras de negócio, limpeza automática ou transformações que modifiquem os dados originais.

---

###### Fonte de dados

O componente recebe o DataFrame `df`, disponibilizado pelo Notebook 01 — Data Ingestion.

A origem pode ser CSV, Parquet, Delta ou tabela Spark, desde que tenha sido carregada como um DataFrame PySpark válido.

O Data Preparation não depende do nome da fonte, das colunas ou do domínio dos dados.

---

###### Principais etapas

1. **Integração:** executa o Data Ingestion e recebe o DataFrame.
2. **Validação:** verifica a existência de colunas e registros.
3. **Identificação do schema:** registra nomes, tipos, posições e nulabilidade.
4. **Classificação:** agrupa as colunas por categorias de tipos Spark.
5. **Aplicabilidade:** identifica as colunas elegíveis para cada Profile.
6. **Configuração:** disponibiliza os parâmetros compartilhados do framework.
7. **Preparação:** disponibiliza `df_prepared` sem alterar os dados originais.
8. **Testes:** executa verificações estruturais opcionais.

---

###### Contrato de entrada

| Entrada | Descrição |
|---|---|
| `df` | DataFrame PySpark produzido pelo Data Ingestion. |
| `df.schema` | Estrutura Spark utilizada na identificação dos tipos. |
| `INGESTION_SOURCE_TYPE` | Tipo da fonte configurada na ingestão, quando disponível. |

---

###### Contrato de saída

**DataFrame principal: `df_prepared`**

Referência padronizada ao DataFrame original, utilizada pelos Profiles seguintes.

**DataFrame auxiliar: `schema_df`**

Contém informações técnicas de cada coluna:

- `posicao`
- `coluna`
- `tipo_spark`
- `tipo_detalhado`
- `nullable`

**Variáveis estruturais:**

- `total_registros`
- `total_colunas`
- `colunas_numericas`
- `colunas_texto`
- `colunas_data`
- `colunas_data_hora`
- `colunas_booleanas`
- `colunas_outros`

**Variáveis de aplicabilidade:**

- `colunas_schema`
- `colunas_distribution`
- `colunas_pattern`
- `colunas_outlier`
- `colunas_correlation`

---

###### Aplicabilidade dos Profiles

| Profile | Colunas elegíveis |
|---|---|
| Schema | Todas as colunas. |
| Distribution | Numéricas, texto, data, data/hora e booleanas. |
| Pattern | Texto. |
| Outlier | Numéricas. |
| Correlation | Numéricas. |

A elegibilidade indica compatibilidade inicial de tipo, não garante que uma análise produzirá resultados estatisticamente significativos.

---

###### Regras de preservação

O Data Preparation não executa:

- Remoção de duplicados;
- Substituição de valores NULL;
- Preenchimento de valores vazios;
- Conversão automática de tipos;
- Exclusão de registros;
- Normalização de valores;
- Tratamento automático de outliers.

Essas características são preservadas para avaliação pelos Profiles especializados.

---

###### Características do componente

- **Genérico:** independente do domínio dos dados.
- **Reutilizável:** compartilha informações estruturais.
- **Não intrusivo:** preserva os registros originais.
- **Escalável:** utiliza DataFrames Spark.
- **Integrável:** mantém contratos padronizados entre notebooks.
- **Rastreável:** disponibiliza informações técnicas sobre o schema.

---

###### DataFrames de resultado

| DataFrame | Finalidade | Granularidade |
|---|---|---|
| `df_prepared` | Entrada padronizada dos Profiles. | Uma linha por registro original. |
| `schema_df` | Inventário técnico das colunas. | Uma linha por coluna. |

---

###### Resultado esperado

Disponibilizar um DataFrame estruturalmente validado, acompanhado das informações de schema, classificação de tipos e listas de aplicabilidade necessárias à execução dos Profiles.

> **Princípio arquitetural:** preparar os metadados para análise sem corrigir ou modificar antecipadamente os problemas de qualidade dos dados.


In [0]:
%run "./01_DATA_INGESTION"

In [0]:
# ============================================================
# 02.3 VALIDAÇÃO DO DATAFRAME
# ============================================================
#
# O QUE FAZ:
# Valida a entrada e contabiliza registros e colunas.
#
# COMO FAZ:
# Verifica o contrato Spark e executa uma contagem distribuída.
#
# POR QUE É IMPORTANTE:
# Impede que Profiles recebam um dataset vazio ou inválido.
#
# PERGUNTA RESPONDIDA:
# "O DataFrame de entrada é adequado à análise?"

if "df" not in globals():
    raise ValueError("O DataFrame 'df' não foi disponibilizado pelo Data Ingestion.")

if not isinstance(df, DataFrame):
    raise TypeError("A entrada 'df' deve ser um DataFrame PySpark.")

if not df.columns:
    raise ValueError("O DataFrame não possui colunas.")

if len(df.columns) != len(set(df.columns)):
    raise ValueError("O DataFrame possui nomes de colunas duplicados.")

if any(not nome.strip() for nome in df.columns):
    raise ValueError("O DataFrame possui nomes de colunas vazios.")

total_colunas = len(df.columns)
total_registros = df.count()

if total_registros == 0:
    raise ValueError("O DataFrame não possui registros.")

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros:,}")
print(f"Total de colunas: {total_colunas}")

In [0]:
# ============================================================
# 02.4 IDENTIFICAÇÃO DO SCHEMA
# ============================================================
#
# O QUE FAZ:
# Disponibiliza os metadados estruturais das colunas.
#
# COMO FAZ:
# Extrai o schema Spark e cria um DataFrame com tipos explícitos.
#
# POR QUE É IMPORTANTE:
# Garante um contrato estável para o catálogo estrutural.
#
# PERGUNTA RESPONDIDA:
# "Quais são os nomes, tipos e posições das colunas?"

schema_info = [
    (
        posicao,
        campo.name,
        campo.dataType.simpleString(),
        campo.dataType.typeName(),
        campo.nullable,
    )
    for posicao, campo in enumerate(df.schema.fields)
]

schema_info_struct = T.StructType([
    T.StructField("posicao", T.IntegerType(), False),
    T.StructField("coluna", T.StringType(), False),
    T.StructField("tipo_spark", T.StringType(), False),
    T.StructField("tipo_detalhado", T.StringType(), False),
    T.StructField("nullable", T.BooleanType(), False),
])

schema_df = spark.createDataFrame(schema_info, schema=schema_info_struct)

print("Schema identificado com sucesso.")
print(f"Colunas catalogadas: {len(schema_info)}")

In [0]:
# ============================================================
# 02.5 CLASSIFICAÇÃO DOS TIPOS DE COLUNA
# ============================================================
#
# O QUE FAZ:
# Classifica os tipos Spark para seleção analítica.
#
# COMO FAZ:
# Consulta os metadados sem ler os valores dos registros.
#
# POR QUE É IMPORTANTE:
# Evita aplicar Profiles incompatíveis com o tipo de coluna.
#
# PERGUNTA RESPONDIDA:
# "Quais categorias técnicas existem no dataset?"

colunas_numericas = []
colunas_texto = []
colunas_data = []
colunas_data_hora = []
colunas_booleanas = []
colunas_outros = []

_timestamp_ntz_type = getattr(T, "TimestampNTZType", None)

for campo in df.schema.fields:
    tipo = campo.dataType
    nome = campo.name

    if isinstance(tipo, NumericType):
        colunas_numericas.append(nome)
    elif isinstance(tipo, StringType):
        colunas_texto.append(nome)
    elif isinstance(tipo, DateType):
        colunas_data.append(nome)
    elif isinstance(tipo, TimestampType) or (
        _timestamp_ntz_type is not None
        and isinstance(tipo, _timestamp_ntz_type)
    ):
        colunas_data_hora.append(nome)
    elif isinstance(tipo, BooleanType):
        colunas_booleanas.append(nome)
    else:
        colunas_outros.append(nome)

print("CLASSIFICAÇÃO DAS COLUNAS")
print("-" * 50)
for categoria, nomes in [
    ("Numéricas", colunas_numericas),
    ("Texto", colunas_texto),
    ("Data", colunas_data),
    ("Data/hora", colunas_data_hora),
    ("Booleanas", colunas_booleanas),
    ("Outros tipos", colunas_outros),
]:
    print(f"{categoria}: {len(nomes)}")

In [0]:
# ============================================================
# 02.6 IDENTIFICAÇÃO DAS COLUNAS ANALISÁVEIS
# ============================================================
#
# O QUE FAZ:
# Define as listas de aplicabilidade inicial dos Profiles.
#
# COMO FAZ:
# Agrupa as categorias técnicas identificadas na célula 02.5.
#
# POR QUE É IMPORTANTE:
# Preserva os contratos consumidos pelos notebooks seguintes.
#
# PERGUNTA RESPONDIDA:
# "Quais colunas são candidatas a cada Profile?"

colunas_distribution = (
    colunas_numericas
    + colunas_texto
    + colunas_data
    + colunas_data_hora
    + colunas_booleanas
)

colunas_pattern = list(colunas_texto)
colunas_outlier = list(colunas_numericas)
colunas_correlation = list(colunas_numericas)
colunas_schema = list(df.columns)

print("APLICABILIDADE DOS PROFILES")
print("-" * 50)
for profile, nomes in [
    ("Schema Profile", colunas_schema),
    ("Distribution Profile", colunas_distribution),
    ("Pattern Profile", colunas_pattern),
    ("Outlier Profile", colunas_outlier),
    ("Correlation Profile", colunas_correlation),
]:
    print(f"{profile}: {len(nomes)} colunas")

In [0]:
# ============================================================
# 02.7 CONFIGURAÇÕES DO FRAMEWORK
# ============================================================
#
# O QUE FAZ:
# Define parâmetros gerais utilizados pelos Profiles.
#
# COMO FAZ:
# Mantém nomes e valores padrão existentes no projeto.
#
# POR QUE É IMPORTANTE:
# Preserva a compatibilidade com as análises seguintes.
#
# PERGUNTA RESPONDIDA:
# "Quais parâmetros compartilhados controlam os Profiles?"

FRAMEWORK_NAME = "Data Quality Framework"

# DISTRIBUTION PROFILE
TOP_N = 5
TOP_CONCENTRATION_N = [1, 3, 5]

# PATTERN PROFILE
PATTERN_TOP_N = 5

# DATA QUALITY SCORE
SCORE_MIN = 0
SCORE_MAX = 100

# EXECUÇÃO E DIAGNÓSTICO
DISPLAY_RESULTS = True
PREPARATION_RUN_TESTS = False

if not isinstance(FRAMEWORK_NAME, str) or not FRAMEWORK_NAME.strip():
    raise ValueError("FRAMEWORK_NAME deve ser uma string não vazia.")

for nome, valor in [
    ("TOP_N", TOP_N),
    ("PATTERN_TOP_N", PATTERN_TOP_N),
]:
    if type(valor) is not int or valor <= 0:
        raise ValueError(f"{nome} deve ser um inteiro positivo.")

if (
    not isinstance(TOP_CONCENTRATION_N, list)
    or not TOP_CONCENTRATION_N
    or any(type(n) is not int or n <= 0 for n in TOP_CONCENTRATION_N)
    or len(TOP_CONCENTRATION_N) != len(set(TOP_CONCENTRATION_N))
):
    raise ValueError("TOP_CONCENTRATION_N deve conter inteiros positivos únicos.")

if not (isinstance(SCORE_MIN, (int, float)) and isinstance(SCORE_MAX, (int, float)) and SCORE_MIN < SCORE_MAX):
    raise ValueError("SCORE_MIN deve ser menor que SCORE_MAX.")

if type(DISPLAY_RESULTS) is not bool or type(PREPARATION_RUN_TESTS) is not bool:
    raise TypeError("As opções de execução devem ser booleanas.")

print("Configurações do framework validadas e carregadas.")

In [0]:
# ============================================================
# 02.8 DATAFRAME PREPARADO
# ============================================================
#
# O QUE FAZ:
# Disponibiliza a entrada padronizada dos Profiles.
#
# COMO FAZ:
# Mantém a referência ao DataFrame recebido da ingestão.
#
# POR QUE É IMPORTANTE:
# Preserva os dados originais para avaliação de qualidade.
#
# PERGUNTA RESPONDIDA:
# "Qual DataFrame será analisado pelos Profiles?"

df_prepared = df

print("DataFrame preparado com sucesso.")
print(f"Registros: {total_registros:,}")
print(f"Colunas: {total_colunas}")

if DISPLAY_RESULTS:
    print("SCHEMA DO DATAFRAME")
    display(schema_df.orderBy("posicao"))
else:
    print("Visualização do schema desabilitada (DISPLAY_RESULTS=False).")

In [0]:
# ============================================================
# 02.9 TESTES OPCIONAIS DO CONTRATO
# ============================================================
#
# O QUE FAZ:
# Verifica invariantes estruturais da preparação.
#
# COMO FAZ:
# Executa testes de metadados, sem nova contagem dos dados.
#
# POR QUE É IMPORTANTE:
# Detecta regressões no contrato de saída.
#
# PERGUNTA RESPONDIDA:
# "A preparação preservou a estrutura da entrada?"

if PREPARATION_RUN_TESTS:
    assert isinstance(df_prepared, DataFrame)
    assert df_prepared is df, "A preparação alterou a referência da entrada."
    assert df_prepared.schema == df.schema
    assert len(schema_df.columns) == 5
    assert len(schema_info) == total_colunas
    assert len(set(colunas_schema)) == total_colunas
    assert (
        len(colunas_numericas)
        + len(colunas_texto)
        + len(colunas_data)
        + len(colunas_data_hora)
        + len(colunas_booleanas)
        + len(colunas_outros)
        == total_colunas
    )
    print("TESTES CONCLUÍDOS: contrato estrutural aprovado.")
else:
    print("Testes desabilitados. Defina PREPARATION_RUN_TESTS=True na célula 02.7.")